# Diffusion Models from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/diffusion_models_from_scratch.ipynb)

Destroying data is easy and needs no model: add a little Gaussian noise, repeat. The surprise is that learning to undo one step of that is enough to generate.

This notebook builds a denoising diffusion model in NumPy with the gradients derived by hand, then measures the thing everyone quotes and few check: what cutting the sampler from 400 network calls to 10 actually costs.

Everything runs on a CPU in about three minutes. No GPU, no framework.

Companion post: [Diffusion Models from Scratch](https://sesen.ai/blog/diffusion-models-from-scratch)

## 1. The data

Two interleaving crescents. Two dimensions means every intermediate state can be drawn, and a generative model's failures are visible rather than inferred.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def two_moons(n, noise=0.08, seed=0):
    """Two interleaving crescents, standardised.

    Every failure mode of a generative model is visible on this data: a missing
    moon, a bridge across the gap that the data does not have, or a cloud that has
    forgotten there was a gap at all."""
    rng = np.random.default_rng(seed)
    m = n // 2
    a = np.linspace(0, np.pi, m)
    outer = np.stack([np.cos(a), np.sin(a)], 1)
    inner = np.stack([1 - np.cos(a), 0.5 - np.sin(a)], 1)
    X = np.concatenate([outer, inner]) + rng.normal(0, noise, (2 * m, 2))
    X = (X - X.mean(0)) / X.std(0)
    return X[rng.permutation(len(X))]     # or every X[:k] is mostly one crescent


X = two_moons(3000)
print("data:", X.shape)

## 2. The forward process

Add a little Gaussian noise `T` times. Because each step is Gaussian, so is the composition of any number of them, which gives the identity the whole method rests on:

```
x_t = sqrt(abar_t) * x_0 + sqrt(1 - abar_t) * eps,    eps ~ N(0, I)
```

Any noise level is one line away from the clean sample. Training never walks the chain.

`abar[-1]` has to reach roughly zero, or the last step is not noise and sampling starts from the wrong distribution.

In [ ]:
T = 400
beta = np.linspace(1e-4 * 1000 / T, 0.02 * 1000 / T, T)   # noise added at step t
alpha = 1 - beta
abar = np.cumprod(alpha)                                  # signal left at step t
abar_prev = np.concatenate([[1.0], abar[:-1]])
print(f"signal left at t = T: {abar[-1]:.1e}")            # must reach ~0


def q_sample(x0, t, eps):
    """The closed form. Any noise level in one line, no chain to walk."""
    return np.sqrt(abar[t])[:, None] * x0 + np.sqrt(1 - abar[t])[:, None] * eps

## 3. The network

A plain MLP over `[x_t, time_features(t)]`. Nothing in it is diffusion-specific: the method's content is in what it is asked to predict, not in its shape.

The backward pass is short enough to write by hand because the loss is a mean squared error, so `d(loss)/d(output)` is `2 * (prediction - target) / n`.

In [ ]:
def time_features(t, dim=32):
    """Sinusoids of the timestep. A raw t is a poor input to an MLP: nearby steps
    should look similar and distant ones separable, which is what this gives."""
    half = dim // 2
    freqs = np.exp(-np.log(10000.0) * np.arange(half) / half)
    ang = np.asarray(t, float)[:, None] * freqs[None, :]
    return np.concatenate([np.cos(ang), np.sin(ang)], 1)


class Dense:
    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out))
        self.b = np.zeros(n_out)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, g):
        self.gW, self.gb = self.x.T @ g, g.sum(0)
        return g @ self.W.T


class Denoiser:
    """eps_hat = f(x_t, t). A plain MLP over [x_t, time_features(t)], SiLU inside.

    Nothing here is diffusion-specific. The method's content is in what the
    network is asked to predict, not in its shape."""

    def __init__(self, dim, hidden=128, depth=3, t_dim=32, seed=0):
        rng = np.random.default_rng(seed)
        self.t_dim = t_dim
        widths = [dim + t_dim] + [hidden] * depth + [dim]
        self.L = [Dense(a, b, rng) for a, b in zip(widths[:-1], widths[1:])]
        self.L[-1].W *= 0.1        # start near zero, a good guess for unit noise

    def forward(self, x, t):
        h = np.concatenate([x, time_features(t, self.t_dim)], 1)
        self.a = []
        for i, layer in enumerate(self.L):
            h = layer.forward(h)
            if i < len(self.L) - 1:
                s = 1 / (1 + np.exp(-np.clip(h, -60, 60)))
                self.a.append((h, s))
                h = h * s
        return h

    def backward(self, g):
        for i in reversed(range(len(self.L))):
            if i < len(self.L) - 1:
                pre, s = self.a[i]
                g = g * (s * (1 + pre * (1 - s)))          # d/dz [z * sigmoid(z)]
            g = self.L[i].backward(g)

    def params_and_grads(self):
        return [(l.W, l.gW) for l in self.L] + [(l.b, l.gb) for l in self.L]

## 4. Training

Three lines do the work: pick a random timestep, pick the noise, jump straight to that noise level. Then a mean squared error against the noise you picked.

There is no chain to simulate, no adversary, no bound to evaluate. This is the part that reads as too simple to work.

In [ ]:
def train(model, X, steps=12000, lr=2e-3, batch=256, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    # Adam state has to be built from the parameters alone: the gradients do not
    # exist until the first backward pass.
    params = [l.W for l in model.L] + [l.b for l in model.L]
    state = [(np.zeros_like(p), np.zeros_like(p)) for p in params]
    for step in range(1, steps + 1):
        x0 = X[rng.integers(0, n, batch)]
        # --- the whole method, three lines ---
        t = rng.integers(0, T, batch)                 # a random noise level
        eps = rng.standard_normal(x0.shape)           # the noise that was added
        xt = q_sample(x0, t, eps)                     # jump straight there
        # -------------------------------------
        diff = model.forward(xt, t) - eps             # ask the net which noise
        model.backward(2 * diff / (batch * d))        # d(MSE)/d(output)

        for (p, g), (m, v) in zip(model.params_and_grads(), state):
            m *= 0.9
            m += 0.1 * g
            v *= 0.999
            v += 0.001 * g**2
            p -= lr * (m / (1 - 0.9**step)) / (np.sqrt(v / (1 - 0.999**step)) + 1e-8)
    return model

## 5. Sampling

Two samplers over the same trained weights.

**Ancestral (DDPM)** walks back down the chain the model was trained on, injecting fresh noise at every step. It needs all `T` network calls.

**DDIM** rewrites the step in terms of the predicted clean sample, which stays valid on any subsequence of timesteps, so it can skip most of them. `eta` scales the injected noise: `eta = 0` is fully deterministic, `eta = 1` recovers ancestral sampling on the subsequence it walks.

In [ ]:
def ddpm_sample(model, n, dim, seed=0):
    """Ancestral sampling: walk back down the chain the model was trained on,
    injecting fresh noise at every step. All T network calls."""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, dim))
    for t in reversed(range(T)):
        eps = model.forward(x, np.full(n, t))
        x = (x - beta[t] / np.sqrt(1 - abar[t]) * eps) / np.sqrt(alpha[t])
        if t > 0:
            var = beta[t] * (1 - abar_prev[t]) / (1 - abar[t])
            x += np.sqrt(var) * rng.standard_normal(x.shape)
    return x


def ddim_sample(model, n, dim, n_steps=50, eta=0.0, seed=0):
    """DDIM: the same trained network, a shorter path.

    The step is rewritten in terms of the predicted clean sample, which stays
    valid on any subsequence of timesteps. eta scales the injected noise: eta = 0
    is deterministic, eta = 1 recovers ancestral sampling on that subsequence."""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, dim))
    taus = np.linspace(0, T - 1, n_steps).round().astype(int)[::-1]
    for i, t in enumerate(taus):
        prev = taus[i + 1] if i + 1 < len(taus) else -1
        ab_t = abar[t]
        ab_prev = abar[prev] if prev >= 0 else 1.0
        eps = model.forward(x, np.full(n, t))
        x0 = (x - np.sqrt(1 - ab_t) * eps) / np.sqrt(ab_t)
        sigma = (eta * np.sqrt((1 - ab_prev) / (1 - ab_t) * (1 - ab_t / ab_prev))
                 if prev >= 0 else 0.0)
        x = np.sqrt(ab_prev) * x0 + np.sqrt(max(1 - ab_prev - sigma**2, 0)) * eps
        if sigma > 0:
            x += sigma * rng.standard_normal(x.shape)
    return x

## 6. Measuring it

"The samples look right" is not a measurement. The energy distance compares two *samples* rather than two points, and it has a floor you can compute: the distance between two independent draws of the real data.

Read the table against that floor, and note which column moves when the step count drops.

In [ ]:
def energy_distance(a, b):
    """A distance between two samples, not between two points. Zero only when the
    distributions match. Unlike eyeing a scatter plot it notices a missing mode or a
    cloud that is too fat, and it needs no density and no fitted judge."""
    from scipy.spatial.distance import cdist
    return 2 * cdist(a, b).mean() - cdist(a, a).mean() - cdist(b, b).mean()


model = train(Denoiser(2), X)
holdout = two_moons(3000, seed=99)
print(f"{'sampler':<24}{'net calls':>10}{'energy':>10}")
print(f"{'real data (the floor)':<24}{'-':>10}{energy_distance(X, holdout):>10.4f}")
runs = [("ancestral", T, ddpm_sample(model, 3000, 2, seed=7))]
for k in (10, 50):
    runs.append((f"DDIM, eta = 0", k, ddim_sample(model, 3000, 2, n_steps=k, seed=7)))
    runs.append((f"DDIM, eta = 1", k, ddim_sample(model, 3000, 2, n_steps=k, eta=1.0, seed=7)))
for label, k, s in runs:
    print(f"{label:<24}{k:>10}{energy_distance(s, holdout):>10.4f}")

## 7. The same thing, drawn

The table says the deterministic sampler is worse. This says where.

In [ ]:
fig, axes = plt.subplots(1, len(runs) + 1, figsize=(3 * (len(runs) + 1), 3.1))
for ax, (label, pts) in zip(axes, [("real data", holdout)] + [(f"{l}, {k}", s)
                                                              for l, k, s in runs]):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.4, linewidths=0)
    ax.set_title(label, fontsize=10)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Exercises

1. **Watch the gap close.** Sweep `eta` from 0 to 1 at 10 steps and plot the energy distance. The injected noise is a dial, not a switch.
2. **Break the schedule.** Replace the linear `beta` with the cosine schedule of Nichol & Dhariwal (2021), `abar_t = cos^2(((t/T + 0.008)/1.008) * pi/2)`, normalised so `abar_0 = 1`. Ancestral sampling barely notices. DDIM falls apart, and the reason is visible in `abar[-1]`.
3. **Predict the sample instead of the noise.** Train with `x0` as the target and recover the noise as `(x_t - sqrt(abar_t) * x0_hat) / sqrt(1 - abar_t)` at sampling time. The two are algebraically the same function. Measure whether they train the same.
4. **Make it conditional.** Append a one-hot label to the network's input, train on data where you know which crescent each point came from, and sample one crescent only. Then interpolate the label between the two, which is guidance in its simplest form.
5. **Move to digits.** Swap in scikit-learn's 8x8 `load_digits`, scaled to [-1, 1], with `hidden=256`. The code above needs no other change. Judge the samples by class coverage rather than by eye.


## Further reading

- Ho, Jain & Abbeel (2020), [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239)
- Sohl-Dickstein et al. (2015), [Deep Unsupervised Learning using Nonequilibrium Thermodynamics](https://arxiv.org/abs/1503.03585), the original formulation
- Song, Meng & Ermon (2021), [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502)
- Nichol & Dhariwal (2021), [Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672), on the cosine schedule
- Ho & Salimans (2022), [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598)
- [Variational Autoencoders from Scratch](https://sesen.ai/blog/variational-autoencoder-from-scratch), the model this one generalises
